In [ ]:
# =========================
# 1. INSTALLS & IMPORTS
# =========================

import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
import textstat
import io
import re
from groq import Groq
import nltk
from nltk.tokenize import sent_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
from PIL import Image  # for BytesIO -> image

nltk.download("punkt")
nltk.download("vader_lexicon")

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")


# =========================
# 2. CORE PLATFORM CLASS
# =========================
class DataGroqAIPlatform:
    def __init__(self):
        # LLM / Groq
        self.groq_client = None
        self.llm_ready = False

        # Data
        self.raw_df = None
        self.cleaned_df = None
        self.dataset_metadata = {}
        self.eda_summary = None
        self.correlation_matrix = None
        self.numeric_summary = None

        # Visualizations
        self.visualization_memory = []

        # ML
        self.ml_results = {
            "classification": None,
            "regression": None
        }

        # Conversation
        self.conversation_history = []

        # Text analytics
        self.text_column = None

        # Sentiment
        self.sia = SentimentIntensityAnalyzer()

    # -------------------------
    # STEP 1: GROQ SETUP
    # -------------------------
    def setup_groq(self, api_key: str):
        if not api_key:
            return "❌ Please provide a valid Groq API key."
        try:
            self.groq_client = Groq(api_key=api_key)
            _ = self.groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": "Test"}],
                max_tokens=5,
            )
            self.llm_ready = True
            return "✅ Groq API connected successfully. LLM ready for conversation."
        except Exception as e:
            self.llm_ready = False
            return f"❌ Error connecting to Groq: {str(e)}"

    # -------------------------
    # STEP 2: DATA UPLOAD
    # -------------------------
    def upload_tabular_dataset(self, file):
        try:
            if file is None:
                return None, "❌ Please upload a CSV or Excel file."

            name = file.name if hasattr(file, "name") else str(file)

            if name.endswith(".csv"):
                df = pd.read_csv(file)
            elif name.endswith(".xlsx") or name.endswith(".xls"):
                df = pd.read_excel(file)
            else:
                return None, "❌ Unsupported format. Use CSV / Excel."

            self.raw_df = df.copy()
            self.cleaned_df = None

            self.dataset_metadata = {
                "shape": df.shape,
                "columns": df.columns.tolist(),
                "dtypes": df.dtypes.astype(str).to_dict(),
                "missing_counts": df.isnull().sum().to_dict(),
                "numeric_columns": df.select_dtypes(include=[np.number]).columns.tolist(),
                "categorical_columns": df.select_dtypes(exclude=[np.number]).columns.tolist(),
            }

            return df, f"✅ Dataset uploaded: {df.shape[0]} rows, {df.shape[1]} columns."
        except Exception as e:
            return None, f"❌ Error reading file: {str(e)}"

    # -------------------------
    # STEP 3: DATA CLEANING
    # -------------------------
    def clean_and_preprocess(self, df: pd.DataFrame):
        if df is None:
            return None, "❌ No dataset loaded."

        try:
            original_shape = df.shape
            df = df.dropna(how="all")

            obj_cols = df.select_dtypes(include=["object"]).columns
            for col in obj_cols:
                df[col] = (
                    df[col]
                    .astype(str)
                    .str.strip()
                    .str.replace(r"\s+", " ", regex=True)
                    .str.replace(r"[^\w\s\.\,\!\?\;\:\-\(\)]", "", regex=True)
                )
                df[col] = df[col].replace("nan", "Not Available").replace("", "Not Available")

            num_cols = df.select_dtypes(include=[np.number]).columns
            if len(num_cols) > 0:
                df[num_cols] = df[num_cols].fillna(df[num_cols].median())

            text_col = None
            for c in df.columns:
                if "text" in c.lower() or "comment" in c.lower() or "review" in c.lower():
                    text_col = c
                    break

            if text_col is not None:
                self.text_column = text_col
                df["text_length"] = df[text_col].astype(str).str.len()
                df["word_count"] = df[text_col].astype(str).str.split().str.len()
                df["sentence_count"] = df[text_col].astype(str).apply(
                    lambda x: len(sent_tokenize(str(x)))
                )
                df["avg_word_length"] = df[text_col].astype(str).apply(
                    lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split()) > 0 else 0
                )
                df["flesch_reading_ease"] = df[text_col].astype(str).apply(
                    lambda x: textstat.flesch_reading_ease(str(x))
                )
                df["flesch_kincaid_grade"] = df[text_col].astype(str).apply(
                    lambda x: textstat.flesch_kincaid_grade(str(x))
                )
                df["sentiment_compound"] = df[text_col].astype(str).apply(
                    lambda x: self.sia.polarity_scores(str(x))["compound"]
                )
                df["sentiment_positive"] = df[text_col].astype(str).apply(
                    lambda x: self.sia.polarity_scores(str(x))["pos"]
                )
                df["sentiment_negative"] = df[text_col].astype(str).apply(
                    lambda x: self.sia.polarity_scores(str(x))["neg"]
                )
                df["sentiment_neutral"] = df[text_col].astype(str).apply(
                    lambda x: self.sia.polarity_scores(str(x))["neu"]
                )

            self.cleaned_df = df.copy()

            cleaning_summary = (
                f"Data Processing Complete:\n"
                f"• Original shape: {original_shape}\n"
                f"• Processed shape: {df.shape}\n"
                f"• Rows removed: {original_shape[0] - df.shape[0]}\n"
                f"• Numeric features: {len(num_cols)}\n"
                f"• Text features: {len(obj_cols)}\n"
                f"• New features added: {df.shape[1] - original_shape[1]}\n"
                f"• Text analytics, readability, and sentiment completed (if text column detected: {text_col})"
            )
            return df, cleaning_summary
        except Exception as e:
            return df, f"❌ Error in cleaning: {str(e)}"

    # -------------------------
    # STEP 4: EDA MODULE
    # -------------------------
    def run_eda(self, df: pd.DataFrame):
        if df is None:
            return "❌ No dataset.", None

        try:
            buf = io.BytesIO()
            fig, axs = plt.subplots(2, 3, figsize=(18, 10))
            axs = axs.flatten()

            # Missing values
            missing = df.isnull().sum()
            missing = missing[missing > 0].sort_values(ascending=False)
            ax = axs[0]
            if len(missing) > 0:
                missing.plot(kind="bar", color="#e74c3c", ax=ax)
                ax.set_title("Missing Values by Column")
                ax.tick_params(axis="x", rotation=45)
            else:
                ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center")
                ax.set_title("Missing Values")

            # Data types
            ax = axs[1]
            dtype_counts = df.dtypes.value_counts()
            ax.pie(dtype_counts.values, labels=dtype_counts.index.astype(str),
                   autopct="%1.1f%%")
            ax.set_title("Data Types")

            # Numeric distribution
            ax = axs[2]
            num_cols = df.select_dtypes(include=[np.number]).columns
            if len(num_cols) > 0:
                col = num_cols[0]
                sns.histplot(df[col], kde=True, ax=ax, color="#3498db")
                ax.set_title(f"Distribution: {col}")
            else:
                ax.text(0.5, 0.5, "No numeric columns", ha="center", va="center")
                ax.set_title("Numeric Distribution")

            # Correlation heatmap
            ax = axs[3]
            if len(num_cols) > 1:
                corr = df[num_cols].corr()
                sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=ax)
                ax.set_title("Correlation Heatmap")
                self.correlation_matrix = corr
                self.visualization_memory.append({
                    "type": "correlation_heatmap",
                    "columns_used": num_cols.tolist(),
                    "summary": "Correlation matrix of numeric features."
                })
            else:
                ax.text(0.5, 0.5, "Not enough numeric columns", ha="center", va="center")
                ax.set_title("Correlation Heatmap")

            # Categorical distribution
            ax = axs[4]
            cat_cols = df.select_dtypes(exclude=[np.number]).columns
            if len(cat_cols) > 0:
                col = cat_cols[0]
                vc = df[col].value_counts().head(10)
                vc.plot(kind="bar", ax=ax, color="#2ecc71")
                ax.set_title(f"Top categories: {col}")
                ax.tick_params(axis="x", rotation=45)
                self.visualization_memory.append({
                    "type": "bar_categories",
                    "columns_used": [col],
                    "summary": f"Top categories in {col}."
                })
            else:
                ax.text(0.5, 0.5, "No categorical columns", ha="center", va="center")
                ax.set_title("Categories")

            # Sentiment distribution
            ax = axs[5]
            if "sentiment_compound" in df.columns:
                sentiment_bins = pd.cut(
                    df["sentiment_compound"],
                    bins=[-1, -0.05, 0.05, 1],
                    labels=["Negative", "Neutral", "Positive"],
                ).value_counts()
                sentiment_bins.plot(kind="bar", ax=ax,
                                    color=["#e74c3c", "#95a5a6", "#27ae60"])
                ax.set_title("Sentiment Distribution")
                self.visualization_memory.append({
                    "type": "sentiment_bar",
                    "columns_used": ["sentiment_compound"],
                    "summary": "Distribution of sentiment categories."
                })
            else:
                ax.text(0.5, 0.5, "No sentiment column", ha="center", va="center")
                ax.set_title("Sentiment Distribution")

            plt.tight_layout()
            plt.savefig(buf, format="png", dpi=150)
            buf.seek(0)

            num_cols = df.select_dtypes(include=[np.number]).columns
            cat_cols = df.select_dtypes(exclude=[np.number]).columns
            if len(num_cols) > 0:
                self.numeric_summary = df[num_cols].describe().to_dict()
            else:
                self.numeric_summary = {}

            basic_info = (
                f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n"
                f"Numeric columns: {len(num_cols)}, Categorical columns: {len(cat_cols)}"
            )
            self.eda_summary = basic_info

            return basic_info, buf
        except Exception as e:
            return f"❌ Error in EDA: {str(e)}", None

    # -------------------------
    # STEP 5: ML MODULE (AUTO)
    # -------------------------
    def run_ml_models(self, df: pd.DataFrame):
        if df is None:
            return "❌ No dataset loaded.", None

        if df.shape[0] < 10:
            return "❌ Dataset too small for ML.", None

        # Auto-select target and features
        target_column = df.columns[-1]
        feature_cols = [c for c in df.columns if c != target_column]

        if target_column not in df.columns:
            return "❌ Invalid target column.", None

        for col in feature_cols:
            if col not in df.columns:
                return f"❌ Feature column '{col}' not found.", None

        try:
            y = df[target_column]
            X = df[feature_cols]

            X = pd.get_dummies(X, drop_first=True)

            if X.shape[1] == 0:
                return "❌ No usable feature columns.", None

            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            # Auto-detect task type
            if y.nunique() <= 15:
                task_type = "classification"
            else:
                task_type = "regression"

            buf = io.BytesIO()
            fig, axs = plt.subplots(2, 2, figsize=(16, 12))
            axs = axs.flatten()

            # Classification
            if task_type == "classification":
                model1 = LogisticRegression(max_iter=1000)
                model2 = RandomForestClassifier(n_estimators=200, random_state=42)

                model1.fit(X_train, y_train)
                model2.fit(X_train, y_train)

                pred1 = model1.predict(X_test)
                pred2 = model2.predict(X_test)

                acc1 = accuracy_score(y_test, pred1)
                acc2 = accuracy_score(y_test, pred2)

                best_model = model1 if acc1 >= acc2 else model2
                best_name = "Logistic Regression" if best_model == model1 else "Random Forest"
                best_acc = max(acc1, acc2)

                # Confusion Matrix
                cm = confusion_matrix(y_test, best_model.predict(X_test))
                sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axs[0])
                axs[0].set_title("Confusion Matrix")

                # Accuracy Comparison
                axs[1].bar(["Logistic", "RandomForest"], [acc1, acc2])
                axs[1].set_ylim(0, 1)
                axs[1].set_title("Accuracy Comparison")

                # Feature Importance
                if hasattr(best_model, "feature_importances_"):
                    importances = best_model.feature_importances_
                    idx = np.argsort(importances)[-10:]
                    axs[2].barh(range(len(idx)), importances[idx])
                    axs[2].set_yticks(range(len(idx)))
                    axs[2].set_yticklabels(np.array(X.columns)[idx])
                    axs[2].set_title("Top Feature Importance")
                else:
                    axs[2].text(0.5, 0.5, "Feature Importance Not Available", ha="center")

                # Precision / Recall / F1
                report = classification_report(
                    y_test,
                    best_model.predict(X_test),
                    output_dict=True
                )
                metrics = ["precision", "recall", "f1-score"]
                values = [report["weighted avg"][m] for m in metrics]
                axs[3].bar(metrics, values)
                axs[3].set_ylim(0, 1)
                axs[3].set_title("Precision / Recall / F1")

                result_text = (
                    f"Classification Results:\n"
                    f"• Auto-detected task: Classification\n"
                    f"• Target: {target_column}\n"
                    f"• Features used: {len(feature_cols)}\n"
                    f"• Best Model: {best_name}\n"
                    f"• Accuracy: {best_acc:.4f}"
                )

                self.ml_results["classification"] = {
                    "best_model": best_name,
                    "accuracy": best_acc,
                }

            # Regression
            else:
                model1 = LinearRegression()
                model2 = RandomForestRegressor(n_estimators=200, random_state=42)

                model1.fit(X_train, y_train)
                model2.fit(X_train, y_train)

                pred1 = model1.predict(X_test)
                pred2 = model2.predict(X_test)

                r2_1 = r2_score(y_test, pred1)
                r2_2 = r2_score(y_test, pred2)

                best_model = model1 if r2_1 >= r2_2 else model2
                best_name = "Linear Regression" if best_model == model1 else "Random Forest"
                best_r2 = max(r2_1, r2_2)

                best_pred = best_model.predict(X_test)

                # Actual vs Predicted
                axs[0].scatter(y_test, best_pred, alpha=0.6)
                axs[0].plot(
                    [y_test.min(), y_test.max()],
                    [y_test.min(), y_test.max()],
                    "r--"
                )
                axs[0].set_title("Actual vs Predicted")

                # R² Comparison
                axs[1].bar(["Linear", "RandomForest"], [r2_1, r2_2])
                axs[1].set_ylim(0, 1)
                axs[1].set_title("R² Comparison")

                # Feature Importance
                if hasattr(best_model, "feature_importances_"):
                    importances = best_model.feature_importances_
                    idx = np.argsort(importances)[-10:]
                    axs[2].barh(range(len(idx)), importances[idx])
                    axs[2].set_yticks(range(len(idx)))
                    axs[2].set_yticklabels(np.array(X.columns)[idx])
                    axs[2].set_title("Top Feature Importance")
                else:
                    axs[2].text(0.5, 0.5, "Feature Importance Not Available", ha="center")

                # Residual Plot
                residuals = y_test - best_pred
                axs[3].scatter(best_pred, residuals, alpha=0.6)
                axs[3].axhline(0, linestyle="--")
                axs[3].set_title("Residual Plot")

                result_text = (
                    f"Regression Results:\n"
                    f"• Auto-detected task: Regression\n"
                    f"• Target: {target_column}\n"
                    f"• Features used: {len(feature_cols)}\n"
                    f"• Best Model: {best_name}\n"
                    f"• R² Score: {best_r2:.4f}"
                )

                self.ml_results["regression"] = {
                    "best_model": best_name,
                    "r2": best_r2,
                }

            plt.tight_layout()
            plt.savefig(buf, format="png", dpi=150)
            buf.seek(0)

            return result_text, buf

        except Exception as e:
            return f"❌ ML Error: {str(e)}", None

    # -------------------------
    # STEP 7: CONTEXT BUILDER
    # -------------------------
    def build_system_context(self):
        if self.cleaned_df is None:
            return "No cleaned dataset."
        df = self.cleaned_df
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        ctx = {
            "rows": df.shape[0],
            "columns": df.shape[1],
            "column_names": df.columns.tolist(),
            "numeric_columns": num_cols,
            "eda_summary": self.eda_summary,
            "ml_results": self.ml_results,
            "visualizations": self.visualization_memory,
        }
        lines = [
            f"Dataset rows: {ctx['rows']}",
            f"Dataset columns: {ctx['columns']}",
            f"Columns: {', '.join(ctx['column_names'][:20])}",
            f"Numeric columns: {', '.join(ctx['numeric_columns'][:20])}",
            f"EDA summary: {ctx['eda_summary']}",
            f"ML results: {ctx['ml_results']}",
            f"Visualization summaries: {ctx['visualizations']}",
        ]
        return "\n".join(lines)

    # -------------------------
    # STEP 8–9: INTENT ANALYSIS + ROUTING
    # -------------------------
    def analyze_intent(self, question: str):
        q = question.lower()
        if any(k in q for k in ["filter", "where", "how many", "count of", "top",
                                "greater than", "less than", "mean of", "average of",
                                "group by", "groupby", "distribution of", "by region"]):
            return "code"
        if any(k in q for k in ["plot", "graph", "chart", "trend", "visualize"]):
            return "code"
        if any(k in q for k in ["model", "prediction", "accuracy", "regression", "classification"]):
            return "ml_reasoning"
        if any(k in q for k in ["insight", "overall", "summary", "explain", "why", "compare"]):
            return "insight"
        return "insight"

    # -------------------------
    # STEP 10–11: PANDAS CODE GENERATION + SAFE EXEC
    # -------------------------
    def generate_pandas_code(self, question: str, previous_error: str = None):
        if not self.llm_ready or self.groq_client is None:
            return None, "❌ LLM not connected. Provide Groq API key."

        if self.cleaned_df is None:
            return None, "❌ No cleaned dataset available."

        columns = ", ".join(self.cleaned_df.columns.tolist())

        system_prompt = (
            "You are a pandas expert working on a DataFrame named df.\n"
            "Generate ONLY Python code using pandas/numpy to answer the user question.\n"
            "Rules:\n"
            "- The DataFrame variable is df (already defined).\n"
            "- Do not import anything.\n"
            "- Do not print.\n"
            "- Do NOT use file I/O, OS, or system access.\n"
            "- Store the final answer in a variable named result.\n"
            "- Only return pure Python code, no explanations or markdown.\n"
            "- If a column name seems slightly wrong, fix it by matching existing columns (case-insensitive).\n"
        )

        user_prompt = (
            f"Dataset columns: {columns}\n"
            f"User question: {question}\n"
        )
        if previous_error:
            user_prompt += f"\nPrevious code error: {previous_error}\nPlease correct the code.\n"

        user_prompt += "Write pandas code that computes the answer and assigns it to `result`."

        try:
            completion = self.groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                max_tokens=500,
            )
            code = completion.choices[0].message.content
            return code, None
        except Exception as e:
            return None, f"❌ Error generating code from LLM: {str(e)}"

    def _truncate_large_result(self, result):
        if isinstance(result, pd.DataFrame):
            if result.shape[0] > 50 or result.shape[1] > 20:
                return result.head(50)
        if isinstance(result, pd.Series):
            if result.shape[0] > 50:
                return result.head(50)
        return result

    def safe_execute_code(self, code: str):
        if self.cleaned_df is None:
            return None, "❌ No cleaned dataset."

        if code is None or not isinstance(code, str):
            return None, "❌ No code to execute."

        code = re.sub(r"```(python)?", "", code).replace("```", "").strip()

        forbidden_patterns = [
            "import ",
            "os.",
            "open(",
            "subprocess",
            "system(",
            "__builtins__",
            "eval(",
            "exec(",
            "while True",
            "for (;;)",
        ]
        lower_code = code.lower()
        for p in forbidden_patterns:
            if p in lower_code:
                return None, "❌ Blocked unsafe code pattern."

        safe_globals = {
            "df": self.cleaned_df.copy(),
            "pd": pd,
            "np": np,
        }
        try:
            exec(code, {"__builtins__": {}}, safe_globals)
            if "result" not in safe_globals:
                return None, "❌ Code executed but `result` variable not found."
            result = safe_globals["result"]
            result = self._truncate_large_result(result)
            return result, None
        except Exception as e:
            return None, str(e)

    # -------------------------
    # STEP 12–13: EXPLANATION / INSIGHT MODE
    # -------------------------
    def explain_result(self, question: str, result):
        if not self.llm_ready or self.groq_client is None:
            return "Result:\n" + str(result)

        context = self.build_system_context()

        try:
            completion = self.groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a data analyst. Explain results in clear business language. "
                            "Avoid code; focus on insights."
                        ),
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Dataset/System context:\n{context}\n\n"
                            f"User question: {question}\n\n"
                            f"Computed result:\n{str(result)}\n\n"
                            f"Explain this result in clear business terms."
                        ),
                    },
                ],
                max_tokens=400,
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"Result:\n{str(result)}\n\n(Explanation failed: {str(e)})"

    def insight_only_answer(self, question: str):
        if not self.llm_ready or self.groq_client is None:
            return "❌ LLM not connected. Cannot generate insights."

        context = self.build_system_context()
        try:
            completion = self.groq_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a senior data analyst. Use the dataset context, EDA, ML "
                            "results, and visualization summaries to answer questions."
                        ),
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Dataset/System context:\n{context}\n\n"
                            f"Question: {question}\n\n"
                            f"Provide a concise but insightful answer."
                        ),
                    },
                ],
                max_tokens=400,
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"❌ Error in insight mode: {str(e)})"

    # -------------------------
    # STEP 14: CONVERSATION MEMORY
    # -------------------------
    def add_conversation_turn(self, question: str, answer: str):
        self.conversation_history.append({"question": question, "answer": answer})
        if len(self.conversation_history) > 10:
            self.conversation_history = self.conversation_history[-10:]

    # -------------------------
    # STEP 16: MAIN CONVERSATION API
    # -------------------------
    def ask(self, question: str):
        if self.cleaned_df is None:
            return {
                "answer": "❌ Please upload and clean a dataset first.",
                "data_result": None,
                "insight": None,
                "confidence": 0.0,
                "latency": None,
            }

        intent = self.analyze_intent(question)

        if intent == "code":
            code, err = self.generate_pandas_code(question)
            if err:
                answer = err
                self.add_conversation_turn(question, answer)
                return {
                    "answer": answer,
                    "data_result": None,
                    "insight": None,
                    "confidence": 0.3,
                    "latency": None,
                }

            result, exec_err = self.safe_execute_code(code)
            if exec_err:
                retry_code, retry_err = self.generate_pandas_code(question, previous_error=exec_err)
                if retry_err:
                    answer = f"❌ Code error: {exec_err}"
                    self.add_conversation_turn(question, answer)
                    return {
                        "answer": answer,
                        "data_result": None,
                        "insight": None,
                        "confidence": 0.3,
                        "latency": None,
                    }
                result, exec_err2 = self.safe_execute_code(retry_code)
                if exec_err2:
                    answer = f"❌ Code error after retry: {exec_err2}"
                    self.add_conversation_turn(question, answer)
                    return {
                        "answer": answer,
                        "data_result": None,
                        "insight": None,
                        "confidence": 0.3,
                        "latency": None,
                    }

            explanation = self.explain_result(question, result)
            self.add_conversation_turn(question, explanation)

            return {
                "answer": explanation,
                "data_result": str(result),
                "insight": explanation,
                "confidence": 0.9,
                "latency": None,
            }

        elif intent in ["ml_reasoning", "insight"]:
            insight = self.insight_only_answer(question)
            self.add_conversation_turn(question, insight)
            return {
                "answer": insight,
                "data_result": None,
                "insight": insight,
                "confidence": 0.8,
                "latency": None,
            }

        else:
            insight = self.insight_only_answer(question)
            self.add_conversation_turn(question, insight)
            return {
                "answer": insight,
                "data_result": None,
                "insight": insight,
                "confidence": 0.6,
                "latency": None,
            }


# =========================
# 3. GRADIO UI
# =========================
platform = DataGroqAIPlatform()


def ui_setup_groq(api_key):
    status = platform.setup_groq(api_key)
    return status


def ui_upload(file):
    df, msg = platform.upload_tabular_dataset(file)
    return df, msg


def ui_clean(df):
    new_df, summary = platform.clean_and_preprocess(df)
    return new_df, summary


def ui_eda(df):
    summary, img_buf = platform.run_eda(df)
    if img_buf is None:
        return summary, None
    img_buf.seek(0)
    img = Image.open(img_buf)
    return summary, np.array(img)


def ui_ml(df):
    if platform.cleaned_df is None:
        return "❌ Please run cleaning first.", None
    text, img_buf = platform.run_ml_models(platform.cleaned_df)
    if img_buf is None:
        return text, None
    img_buf.seek(0)
    img = Image.open(img_buf)
    return text, np.array(img)


def ui_converse(question):
    resp = platform.ask(question)
    return resp["answer"], resp["data_result"], resp["insight"], resp["confidence"]


with gr.Blocks() as demo:
    gr.Markdown(
        "<h2 style='text-align:center'>DataGroq AI – End‑to‑End Conversational Data Intelligence Platform</h2>"
    )

    with gr.Tab("Setup & Data"):
        groq_key = gr.Textbox(label="Groq API Key", type="password")
        groq_status = gr.Textbox(label="Groq Status", interactive=False)
        connect_btn = gr.Button("Connect Groq")
        connect_btn.click(ui_setup_groq, inputs=groq_key, outputs=groq_status)

        gr.Markdown("---")

        file_in = gr.File(label="Upload CSV / Excel")
        data_status = gr.Textbox(label="Data Status", interactive=False)
        df_view = gr.Dataframe(label="Dataset Preview", max_height=300)
        file_in.upload(ui_upload, inputs=file_in, outputs=[df_view, data_status])

        gr.Markdown("### Clean & Preprocess")
        clean_btn = gr.Button("Run Cleaning Pipeline")
        cleaning_summary = gr.Textbox(label="Cleaning Summary", interactive=False, lines=5)
        clean_btn.click(ui_clean, inputs=df_view, outputs=[df_view, cleaning_summary])

    with gr.Tab("EDA & Visualizations"):
        eda_btn = gr.Button("Run EDA")
        eda_summary = gr.Textbox(label="EDA Summary", interactive=False, lines=4)
        eda_img = gr.Image(label="EDA Plots", height=420)
        eda_btn.click(ui_eda, inputs=df_view, outputs=[eda_summary, eda_img])

    with gr.Tab("Machine Learning"):
        gr.Markdown(
            "Click **Run ML**. The app will auto-select the last column as target and the rest as features."
        )
        ml_btn = gr.Button("Run ML")
        ml_text = gr.Textbox(label="ML Results", interactive=False, lines=10)
        ml_img = gr.Image(label="ML Evaluation", height=420)
        ml_btn.click(
            ui_ml,
            inputs=[df_view],
            outputs=[ml_text, ml_img],
        )

    with gr.Tab("Conversational Agent"):
        gr.Markdown(
            "<div style='text-align:center'>"
            "Ask anything about your data. The agent will decide whether to run pandas code "
            "or provide high‑level insights."
            "</div>"
        )
        q_box = gr.Textbox(label="Your Question", lines=3)
        answer_box = gr.Textbox(label="Answer", interactive=False, lines=6)
        data_result_box = gr.Textbox(
            label="Raw Data Result (stringified, truncated)", interactive=False, lines=4
        )
        insight_box = gr.Textbox(label="Insight", interactive=False, lines=4)
        conf_box = gr.Number(label="Confidence", interactive=False)
        ask_btn = gr.Button("Ask", variant="primary")

        ask_btn.click(
            ui_converse,
            inputs=q_box,
            outputs=[answer_box, data_result_box, insight_box, conf_box],
        )

demo.launch()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d578b724cdfa27454.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
